In [ ]:
pip install --upgrade pymilvus
pip install "pymilvus[model]"
pip install sentence-transformers
pip install langchain-text-splitters
pip install langchain-openai
pip install langchain-community 
pip install scipy
pip install nltk

In [ ]:
import uuid
from tqdm import tqdm
from pymilvus import DataType, AnnSearchRequest, RRFRanker, WeightedRanker
from pymilvus.model.sparse import BM25EmbeddingFunction
from pymilvus.model.sparse.bm25 import Analyzer
from pymilvus.model.sparse.bm25.tokenizers import build_default_analyzer

from pymilvus import MilvusClient

import nltk

import scipy.sparse as sp

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

import re
import json
from langchain_openai import ChatOpenAI

In [ ]:
# ============================================================
# CONFIGURATION 
# ============================================================

PDF_PATH        = "./data/sample_employee_handbook.pdf" # path of you document
COLLECTION_NAME = "rag_documents_hybrid"
MILVUS_DB_PATH  = "./db/milvus_demo.db"
API_KEY         = "sk-..."
EMBEDDING_MODEL = "text-embedding-3-large"
EMBEDDING_DIM   = 1024
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 100
TOP_K           = 5

In [ ]:
# ============================================================
# STEP 1: INITIALIZE EMBEDDING MODEL
# ============================================================

# Dense embedding model here we'll be using OpenAI's embedding model
embedding_obj = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=API_KEY,
    dimensions=EMBEDDING_DIM
)

# ============================================================
# STEP 2: CREATE MILVUS CONNECTION
# ============================================================

# Milvus Lite - single file, no server needed
client = MilvusClient(MILVUS_DB_PATH)
print("Models and DB connection ready.")

In [ ]:
# ============================================================
# STEP 3: LOAD PDF
# ============================================================

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print(f"Loaded {len(documents)} pages")

In [ ]:
# ============================================================
# STEP 4: CHUNKING
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

In [ ]:
# ============================================================
# STEP 5.1: CREATE DENSE EMBEDDINGS
# ============================================================

texts = [doc.page_content for doc in chunks]

print("Generating dense embeddings...")
dense_embeddings = embedding_obj.embed_documents(texts)
embedding_dim = len(dense_embeddings[0])
print(f"Dense embedding dimension: {embedding_dim}")

# ============================================================
# STEP 5.2: CREATE SPARSE EMBEDDINGS
# ============================================================

print("Fitting BM25 Analyzer for sparse embeddings...")
analyzer = build_default_analyzer(language="en")
bm25_ef = BM25EmbeddingFunction(analyzer)
bm25_ef.fit(texts)  # Always fit on current corpus

sparse_embeddings = bm25_ef.encode_documents(texts)

In [ ]:
# ============================================================
# STEP 6: CREATE HYBRID COLLECTION & INDEXES
# ============================================================

existing_collections = client.list_collections()

if COLLECTION_NAME in existing_collections:
    print(f"Collection '{COLLECTION_NAME}' already exists — dropping and recreating to ensure fresh data...")
    client.drop_collection(COLLECTION_NAME)  # Drop stale collection to avoid empty result bugs

schema = client.create_schema()

schema.add_field(
    field_name="id",
    datatype=DataType.VARCHAR,
    is_primary=True,
    max_length=100
)
schema.add_field(
    field_name="vector",
    datatype=DataType.FLOAT_VECTOR,
    dim=embedding_dim
)
schema.add_field(
    field_name="sparse_vector",
    datatype=DataType.SPARSE_FLOAT_VECTOR
)
schema.add_field(
    field_name="text",
    datatype=DataType.VARCHAR,
    max_length=65535
)
schema.add_field(
    field_name="page_number",
    datatype=DataType.INT64
)
schema.add_field(
    field_name="source",
    datatype=DataType.VARCHAR,
    max_length=500
)
schema.add_field(
    field_name="chunk_id",
    datatype=DataType.INT64
)

client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema
)
print("Collection created with hybrid schema configurations.")

# Create indexes
print("Creating indexes...")
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="vector",
    index_type="FLAT",
    metric_type="COSINE"
)
index_params.add_index(
    field_name="sparse_vector",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="IP"
)

client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)
print("Dense and Sparse indexes built successfully.")

In [ ]:
# ============================================================
# STEP 7: PREPARE DATA FOR INSERTION
# ============================================================

def sparse_to_dict(s_emb):
    """Safely convert any scipy sparse row to Milvus-compatible dict."""
    if sp.issparse(s_emb):
        # Convert to coo directly without reshaping
        coo = s_emb.tocoo()
        return {int(col): float(val) for col, val in zip(coo.col, coo.data)}
    elif hasattr(s_emb, 'toarray'):
        # 1D dense array from sparse iteration
        arr = s_emb.toarray().flatten() if hasattr(s_emb, 'toarray') else s_emb
        return {int(i): float(v) for i, v in enumerate(arr) if v != 0.0}
    elif isinstance(s_emb, dict):
        return s_emb
    else:
        # Numpy 1D array fallback
        return {int(i): float(v) for i, v in enumerate(s_emb) if v != 0.0}

data = []
# ✅ Don't use list() — slice rows directly from the 2D sparse matrix
for idx, (chunk, d_emb) in enumerate(
        tqdm(zip(chunks, dense_embeddings), total=len(chunks))
):
    page_no = chunk.metadata.get("page", -1)
    
    # ✅ Slice row as a proper 2D sparse matrix row (shape: 1 x vocab_size)
    s_emb = sparse_embeddings[idx]  
    sparse_vector_data = sparse_to_dict(s_emb)

    if not sparse_vector_data:
        print(f"Warning: Empty sparse vector at chunk {idx}, skipping...")
        continue

    data.append({
        "id": str(uuid.uuid4()),
        "vector": d_emb,
        "sparse_vector": sparse_vector_data,
        "text": str(chunk.page_content),
        "page_number": int(page_no),
        "source": str(PDF_PATH),
        "chunk_id": int(idx)
    })

print(f"Prepared {len(data)} records for insertion.")

In [ ]:
# ============================================================
# STEP 8: INSERT INTO MILVUS
# ============================================================

res = client.insert(
    collection_name=COLLECTION_NAME,
    data=data
)
print(f"Insertion completed. Inserted: {res['insert_count']} records.")

# ✅ CRITICAL: Load collection into memory before searching
print("Loading collection into memory...")
client.load_collection(COLLECTION_NAME)
print(f"Load state: {client.get_load_state(COLLECTION_NAME)}")

In [ ]:
def ensure_loaded(client, collection_name):
    state = client.get_load_state(collection_name)
    if state["state"] != "Loaded":
        print(f"Loading collection '{collection_name}'...")
        client.load_collection(collection_name)
    else:
        print(f"Collection '{collection_name}' already loaded.")

# Call this before any search/query operation
ensure_loaded(client, COLLECTION_NAME)

In [ ]:
# ============================================================
# DENSE VECTOR SEARCH EXECUTION
# ============================================================

query = "leave policy?"
print(f"\nProcessing dense vector search for query: '{query}'")

# Generate only the dense query vector matrix
query_dense_embedding = embedding_obj.embed_query(query)

# Execute standalone vector similarity search on the dense field
results = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_dense_embedding],      # 2D array wrapping your query embedding
    anns_field="vector",              # Targets your original dense vector field
    search_param={"metric_type": "COSINE"},   # Distance metric configuration
    limit=TOP_K,
    output_fields=["text", "page_number", "source"]
)


# DISPLAY RESULTS


print("\nVECTOR SEARCH RETRIEVAL RESULTS (DENSE ONLY)")
print("=" * 50)

if not results or not results[0]:
    print("No results returned.")
else:
    for idx, hit in enumerate(results[0], start=1):
        entity = hit["entity"]
        print(f"\nRank: {idx}")
        # Note: In standard search(), hit['distance'] returns the raw Cosine metric score,
        # unlike hybrid_search() which returns the combined RRF ranking score.
        print(f"Cosine Similarity Score: {hit['distance']:.4f}")
        print(f"Page: {entity['page_number']}")
        print(f"Text:\n{entity['text'][:500]}")

In [ ]:
# ============================================================
# HYBRID SEARCH EXECUTION
# ============================================================

query = "leave policy?"
print(f"\nProcessing hybrid search for query: '{query}'")

# Dense query vector
query_dense_embedding = embedding_obj.embed_query(query)

# Sparse query vector
query_sparse_raw = bm25_ef.encode_queries([query])
sparse_dict = sparse_to_dict(query_sparse_raw[0])  # reuse same helper

print(f"Sparse query terms count: {len(sparse_dict)}")  # Should be > 0

# ✅ Sanity check — abort early if sparse dict is empty
if not sparse_dict:
    print("WARNING: Sparse query dict is empty — BM25 found no matching vocabulary.")
    print("Falling back to dense-only search...")
    results = client.search(
        collection_name=COLLECTION_NAME,
        data=[query_dense_embedding],
        anns_field="vector",
        param={"metric_type": "COSINE"},
        limit=TOP_K,
        output_fields=["text", "page_number", "source"]
    )
else:
    dense_req = AnnSearchRequest(
        data=[query_dense_embedding],
        anns_field="vector",
        param={"metric_type": "COSINE"},
        limit=TOP_K
    )
    sparse_req = AnnSearchRequest(
        data=[sparse_dict],
        anns_field="sparse_vector",
        param={"metric_type": "IP"},
        limit=TOP_K
    )

    results = client.hybrid_search(
        collection_name=COLLECTION_NAME,
        reqs=[dense_req, sparse_req],
        ranker=RRFRanker(k=60),
        limit=TOP_K,
        output_fields=["text", "page_number", "source"]
    )


# DISPLAY RESULTS


print("\nHYBRID RETRIEVAL RESULTS (DENSE + SPARSE)")
print("=" * 50)

if not results or not results[0]:
    print("No results returned.")
else:
    for idx, hit in enumerate(results[0], start=1):
        entity = hit["entity"]
        print(f"\nRank: {idx}")
        print(f"Score: {hit['distance']:.4f}")
        print(f"Page: {entity['page_number']}")
        print(f"Text:\n{entity['text'][:500]}")

In [ ]:
# ============================================================
# RERANKING WITH CROSS-ENCODER
# ============================================================

from sentence_transformers import CrossEncoder

# Load cross-encoder from your local saved path
CROSS_ENCODER_PATH = "./cross-encoder/ms-marco-MiniLM-L12-v2"  # <-- update this
cross_encoder = CrossEncoder(CROSS_ENCODER_PATH)

query = "What is the leave policy?"

# ── Step 1: Broad vector retrieval (fetch more than needed) ──
RETRIEVAL_K = 20   # fetch wide
FINAL_K     = 5    # rerank down to this

In [ ]:
query_dense_embedding = embedding_obj.embed_query(query)

results = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_dense_embedding],      # 2D array wrapping your query embedding
    anns_field="vector",              # Targets your original dense vector field
    search_param={"metric_type": "COSINE"},   # Distance metric configuration
    limit=RETRIEVAL_K,
    output_fields=["text", "page_number", "source"]
)



In [ ]:
hits = results[0]
print(f"Retrieved {len(hits)} chunks for reranking...")

# ── Step 2: Rerank using cross-encoder ──
query_chunk_pairs = [[query, hit["entity"]["text"]] for hit in hits]
rerank_scores = cross_encoder.predict(query_chunk_pairs)

# ── Step 3: Attach scores and sort ──
for hit, score in zip(hits, rerank_scores):
    hit["rerank_score"] = float(score)

reranked = sorted(hits, key=lambda x: x["rerank_score"], reverse=True)[:FINAL_K]

# ── Step 4: Display ──
print("\nRERANKED RESULTS")
print("=" * 50)

for idx, hit in enumerate(reranked, start=1):
    entity = hit["entity"]
    print(f"\nRank     : {idx}")
    print(f"Rerank Score  : {hit['rerank_score']:.4f}")
    print(f"Vector Score  : {hit['distance']:.4f}")
    print(f"Page     : {entity['page_number']}")
    print(f"Text:\n{entity['text'][:500]}")

In [ ]:
print("Fitting BM25 Analyzer for sparse embeddings...")
analyzer = build_default_analyzer(language="en")
bm25_ef = BM25EmbeddingFunction(analyzer)

In [ ]:
# ─────────────────────────────────────────────────────────────
# LLM SETUP  (shared across techniques 2, 3, 4)
# ─────────────────────────────────────────────────────────────

llm = ChatOpenAI(
     model="gpt-4o",
     temperature=0,
     api_key=API_KEY
)

# METADATA FILTERING

What it does:
  Adds a structured pre-filter on scalar fields (page_number, source, etc.)
  BEFORE the vector search runs, narrowing the search space.

When to use:
  - User asks about a specific section, chapter, or date range
  - Multi-tenant setups where users should only see their own docs
  - Reducing latency by limiting candidate pool

How it works in Milvus:
  Pass a `filter` expression string alongside the ANN requests.
  Milvus evaluates the scalar filter first, then runs vector search
  only on matching rows — no post-filtering, so recall is preserved.

In [ ]:
# ════════════════════════════════════════════════════════════
# METADATA FILTERING
# ════════════════════════════════════════════════════════════

def search_with_metadata_filter(
    client, collection_name, embedding_obj, bm25_ef,
    query: str,
    page_range: tuple = None,
    source_file: str = None,
    top_k: int = 5
):
    filter_parts = []

    if page_range:
        lo, hi = page_range
        filter_parts.append(f"page_number >= {lo} && page_number <= {hi}")

    if source_file:
        filter_parts.append(f'source == "{source_file}"')

    filter_expr = " && ".join(filter_parts) if filter_parts else None

    print(f"\n[Metadata Filter] Query : '{query}'")
    print(f"[Metadata Filter] Filter: {filter_expr or 'None (unfiltered)'}")

    results = hybrid_search(
        client, collection_name, embedding_obj, bm25_ef,
        query_text=query,
        top_k=top_k,
        filters=filter_expr
    )
    return results


# ── Run ──────────────────────────────────────────────────────
meta_results = search_with_metadata_filter(
    client, COLLECTION_NAME, embedding_obj, bm25_ef,
    query      = "What is the leave policy?",
    page_range = (1, 30),
    source_file= None,
    top_k      = 5
)

# ── Print Results ─────────────────────────────────────────────
print("\nMETADATA-FILTERED RESULTS")
print("=" * 55)
if not meta_results or not meta_results[0]:
    print("No results returned.")
else:
    for idx, hit in enumerate(meta_results[0], start=1):
        entity = hit["entity"]
        print(f"\nRank  : {idx}")
        print(f"Score : {hit['distance']:.4f}")
        print(f"Page  : {entity['page_number']}")
        print(f"Text  :\n{entity['text'][:400]}")

# QUERY REWRITING / TRANSFORMATION

What it does:
  Uses an LLM to rephrase the user's colloquial query into formal, 
  keyword-rich language that better matches document vocabulary.

Why it helps:
  "how many days off do I get?" → 
  "annual leave entitlement days employee vacation policy"

Strategy used here — generate N rewritten variants, search with each,
then merge and deduplicate results (Reciprocal Rank Fusion over variants).


In [ ]:
# ════════════════════════════════════════════════════════════
# QUERY REWRITING
# ════════════════════════════════════════════════════════════

import re, json

REWRITE_PROMPT = """You are an expert at reformulating search queries to improve document retrieval.

Given a user query, produce {n} alternative search queries that:
- Use formal, document-style language
- Include relevant keywords and synonyms
- Cover different angles of the same question

User query: {query}

Respond ONLY with a JSON array of strings. Example:
["rewritten query 1", "rewritten query 2", "rewritten query 3"]"""


def rewrite_query(query: str, n: int = 3) -> list[str]:
    prompt   = REWRITE_PROMPT.format(query=query, n=n)
    response = llm.invoke(prompt)
    raw      = re.sub(r"^```json|^```|```$", "", response.content.strip(), flags=re.MULTILINE).strip()
    try:
        variants = json.loads(raw)
        return [query] + variants       # always keep the original
    except json.JSONDecodeError:
        print("Warning: Could not parse rewrites, using original query only.")
        return [query]


def search_with_query_rewriting(
    client, collection_name, embedding_obj, bm25_ef,
    query: str,
    n_rewrites: int = 3,
    top_k: int = 5
):
    variants = rewrite_query(query, n=n_rewrites)

    print(f"\n[Query Rewriting] Original  : '{query}'")
    for i, v in enumerate(variants[1:], 1):
        print(f"[Query Rewriting] Variant {i} : '{v}'")

    seen_ids    = {}
    rank_scores = {}

    for variant in variants:
        results = hybrid_search(
            client, collection_name, embedding_obj, bm25_ef,
            query_text=variant,
            top_k=top_k
        )
        if not results or not results[0]:
            continue
        for rank, hit in enumerate(results[0], start=1):
            hit_id = hit["id"]
            rank_scores[hit_id] = rank_scores.get(hit_id, 0) + 1.0 / (60 + rank)
            if hit_id not in seen_ids:
                seen_ids[hit_id] = hit

    merged = sorted(seen_ids.values(), key=lambda h: rank_scores[h["id"]], reverse=True)[:top_k]
    return [merged]


# ── Run ──────────────────────────────────────────────────────
rewrite_results = search_with_query_rewriting(
    client, COLLECTION_NAME, embedding_obj, bm25_ef,
    query      = "What is the leave policy?",
    n_rewrites = 3,
    top_k      = 5
)

# ── Print Results ─────────────────────────────────────────────
print("\nQUERY-REWRITTEN RESULTS")
print("=" * 55)
if not rewrite_results or not rewrite_results[0]:
    print("No results returned.")
else:
    for idx, hit in enumerate(rewrite_results[0], start=1):
        entity = hit["entity"]
        print(f"\nRank  : {idx}")
        print(f"Score : {hit['distance']:.4f}")
        print(f"Page  : {entity['page_number']}")
        print(f"Text  :\n{entity['text'][:400]}")

# HyDE (Hypothetical Document Embeddings)

What it does:
  Instead of embedding the short question, ask the LLM to hallucinate
  a plausible answer document, then embed THAT.

Why it helps:
  - A question and its answer occupy very different positions in vector space
  - A hypothetical answer document lands near real answer documents
  - Dramatically improves recall for short, vague, or jargon-free queries

Caveat:
  The hallucinated document may contain false facts — that's fine, because
  we only use its EMBEDDING, not its content, for retrieval.

In [ ]:
# ════════════════════════════════════════════════════════════
# HyDE (Hypothetical Document Embeddings)
# ════════════════════════════════════════════════════════════

HYDE_PROMPT = """You are a corporate policy document writer.

Write a 2-3 paragraph excerpt from an official HR policy or company document
that would DIRECTLY ANSWER the following question.
Write in formal document style. Do not mention the question itself.

Question: {query}

Document excerpt:"""


def generate_hypothetical_document(query: str) -> str:
    response = llm.invoke(HYDE_PROMPT.format(query=query))
    return response.content.strip()


def search_with_hyde(
    client, collection_name, embedding_obj, bm25_ef,
    query: str,
    top_k: int = 5
):
    hypothetical_doc = generate_hypothetical_document(query)

    print(f"\n[HyDE] Query            : '{query}'")
    print(f"[HyDE] Hypothetical doc :\n  {hypothetical_doc[:300]}...\n")

    # Search using the hypothetical document's embedding
    hyde_results = hybrid_search(
        client, collection_name, embedding_obj, bm25_ef,
        query_text=hypothetical_doc,    # embed the answer, not the question
        top_k=top_k
    )

    # Also search with the original query and merge both via RRF
    original_results = hybrid_search(
        client, collection_name, embedding_obj, bm25_ef,
        query_text=query,
        top_k=top_k
    )

    seen_ids    = {}
    rank_scores = {}

    for result_set in [hyde_results, original_results]:
        if not result_set or not result_set[0]:
            continue
        for rank, hit in enumerate(result_set[0], start=1):
            hit_id = hit["id"]
            rank_scores[hit_id] = rank_scores.get(hit_id, 0) + 1.0 / (60 + rank)
            if hit_id not in seen_ids:
                seen_ids[hit_id] = hit

    merged = sorted(seen_ids.values(), key=lambda h: rank_scores[h["id"]], reverse=True)[:top_k]
    return [merged]


# ── Run ──────────────────────────────────────────────────────
hyde_results = search_with_hyde(
    client, COLLECTION_NAME, embedding_obj, bm25_ef,
    query = "What is the leave policy?",
    top_k = 5
)

# ── Print Results ─────────────────────────────────────────────
print("\nHyDE RESULTS")
print("=" * 55)
if not hyde_results or not hyde_results[0]:
    print("No results returned.")
else:
    for idx, hit in enumerate(hyde_results[0], start=1):
        entity = hit["entity"]
        print(f"\nRank  : {idx}")
        print(f"Score : {hit['distance']:.4f}")
        print(f"Page  : {entity['page_number']}")
        print(f"Text  :\n{entity['text'][:400]}")

# QUERY DECOMPOSITION

What it does:
  Breaks a complex multi-part question into atomic sub-questions,
  retrieves separately for each, then merges all results.

Why it helps:
  "What is the leave policy and how does performance review affect salary?"
  → Two completely different retrieval targets — a single search misses one.

Two strategies:
  a) Sequential  : sub-question N can reference answers from sub-question N-1
  b) Parallel    : all sub-questions searched independently (used here — simpler)

In [ ]:
# ════════════════════════════════════════════════════════════
# QUERY DECOMPOSITION
# ════════════════════════════════════════════════════════════

DECOMPOSE_PROMPT = """You are an expert at breaking down complex questions for document retrieval.

Decompose the following question into 2-4 simple, self-contained sub-questions.
Each sub-question should target a single distinct piece of information.

Complex question: {query}

Respond ONLY with a JSON array of strings. Example:
["sub-question 1", "sub-question 2", "sub-question 3"]"""


def decompose_query(query: str) -> list[str]:
    response = llm.invoke(DECOMPOSE_PROMPT.format(query=query))
    raw      = re.sub(r"^```json|^```|```$", "", response.content.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print("Warning: Could not parse decomposition, using original query.")
        return [query]


def search_with_decomposition(
    client, collection_name, embedding_obj, bm25_ef,
    query: str,
    top_k: int = 5
):
    sub_questions = decompose_query(query)

    print(f"\n[Decomposition] Original query : '{query}'")
    for i, sq in enumerate(sub_questions, 1):
        print(f"[Decomposition] Sub-question {i} : '{sq}'")

    per_subquery_results = {}
    seen_ids             = {}
    rank_scores          = {}

    for sq in sub_questions:
        results = hybrid_search(
            client, collection_name, embedding_obj, bm25_ef,
            query_text=sq,
            top_k=top_k
        )
        per_subquery_results[sq] = results

        if not results or not results[0]:
            continue
        for rank, hit in enumerate(results[0], start=1):
            hit_id = hit["id"]
            rank_scores[hit_id] = rank_scores.get(hit_id, 0) + 1.0 / (60 + rank)
            if hit_id not in seen_ids:
                seen_ids[hit_id] = hit

    merged = sorted(seen_ids.values(), key=lambda h: rank_scores[h["id"]], reverse=True)[:top_k]

    # Per sub-question breakdown
    print("\n── Per Sub-question Results ──")
    for sq, res in per_subquery_results.items():
        print(f"\n  SUB-QUERY: '{sq[:60]}'")
        if res and res[0]:
            for i, hit in enumerate(res[0], start=1):
                print(f"    {i}. Page {hit['entity']['page_number']} | Score {hit['distance']:.4f} | {hit['entity']['text'][:150]}")

    return {"per_subquery": per_subquery_results, "merged": [merged]}


# ── Run ──────────────────────────────────────────────────────
decomp_results = search_with_decomposition(
    client, COLLECTION_NAME, embedding_obj, bm25_ef,
    query = "What is the leave policy and how does it affect salary deductions?",
    top_k = 5
)

# ── Print Merged Results ──────────────────────────────────────
print("\nDECOMPOSED — MERGED FINAL RESULTS")
print("=" * 55)
merged_hits = decomp_results["merged"]
if not merged_hits or not merged_hits[0]:
    print("No results returned.")
else:
    for idx, hit in enumerate(merged_hits[0], start=1):
        entity = hit["entity"]
        print(f"\nRank  : {idx}")
        print(f"Score : {hit['distance']:.4f}")
        print(f"Page  : {entity['page_number']}")
        print(f"Text  :\n{entity['text'][:400]}")